# Mixing SMIRNOFF and atom-typed force fields

The purpose of this example is to show OpenFF infrastructure can prepare a pipeline currently `openmmforcefields` and a mix of SMIRNOFF and atom-typed force fields.

## Data sources

* [Protein PDB](https://github.com/omsf/joint-demo/blob/8fe4145dd3255c961d04bb16c00f39ecd768dc71/source/openfe/protein.pdb)
* [Ligand(s) SDF](https://github.com/omsf/joint-demo/blob/8fe4145dd3255c961d04bb16c00f39ecd768dc71/source/openfe/ligands.sdf)

In [32]:
import openmm.app
import openmm.unit
import openff.toolkit
import openff.interchange
from openff.units.openmm import from_openmm

class ForceFieldTopologyPairing:
    def __init__(
        self,
        forcefields: openmm.app.ForceField | openff.toolkit.ForceField,
        topology: openmm.app.Topology | openff.toolkit.Topology,
        positions: openff.toolkit.Quantity | None = None,
    ):
        self.forcefields = forcefields
        self.topology = topology

        if positions is None:
            self.positions = None
        elif isinstance(positions, openff.toolkit.Quantity):
            self.positions= positions
        else:
            self.positions = from_openmm(positions)

    def make_interchange(self) -> openff.interchange.Interchange:
        if isinstance(self.topology, openff.toolkit.Topology):
            if all(isinstance(f, openff.toolkit.ForceField) for f in self.forcefields):
                merged_force_field = self.forcefields[0]

                for force_field in self.forcefields[1:]:
                    merged_force_field = merged_force_field.merge(force_field)

                return merged_force_field.create_interchange(self.topology)

            elif all(isinstance(f, openmm.app.ForceField) for f in self.forcefields):
                topology = self.topology.to_openmm()

                assert len(self.forcefields) == 1, "OpenMM force fields cannot be combined"
                
                system = self.forcefields[0].createSystem(
                    topology,
                    nonbondedMethod=openmm.app.PME if topology.getPeriodicBoxVectors() is not None else openmm.app.NoCutoff,
                    nonbondedCutoff=9.0 * openmm.unit.angstrom,
                    switchDistance=8.0 * openmm.unit.angstrom,
                )

                return openff.interchange.Interchange.from_openmm(
                    system=system,
                    topology=topology,
                    positions=self.positions,
                )

        elif isinstance(self.topology, openmm.app.Topology):
            if all(isinstance(f, openmm.app.ForceField) for f in self.forcefields):
                assert len(self.forcefields) == 1, "OpenMM force fields cannot be combined"
                
                return openff.interchange.Interchange.from_openmm(
                    system = self.forcefields[0].createSystem(
                        self.topology,
                        nonbondedMethod=openmm.app.PME if self.topology.getPeriodicBoxVectors() is not None else openmm.app.NoCutoff,
                        nonbondedCutoff=9.0 * openmm.unit.angstrom,
                        switchDistance=8.0 * openmm.unit.angstrom,
                    ),
                    topology=self.topology,
                    positions=self.positions,
                )
            
            elif all(isinstance(f, openff.toolkit.ForceField) for f in self.forcefields):
                raise ValueError("Cannot parametrize an OpenMM Topology with SMIRNOFF force fields.")
    
        raise ValueError("Failed.")

class Blob:
    pairings: list[ForceFieldTopologyPairing] = list()

    def __init__(self, pairings: list[ForceFieldTopologyPairing]):
        self.pairings = pairings

    def make_interchange(self) -> openff.interchange.Interchange:
        out = self.pairings[0].make_interchange()

        for pairing in self.pairings[1:]:
            out = out.combine(pairing.make_interchange())

        return out

In [33]:
ligands = openff.toolkit.Molecule.from_file("ligands.sdf")

protein = openmm.app.PDBFile("protein.pdb")

In [40]:
blob = Blob(
    pairings = [
        ForceFieldTopologyPairing(
            forcefields=[openff.toolkit.ForceField("openff-2.2.1.offxml")],
            topology=ligands[-1].to_topology(),
        ),
        ForceFieldTopologyPairing(
            forcefields=[openmm.app.ForceField("amber14-all.xml", "amber14/tip3p.xml")],
            topology=protein.topology,
            positions=protein.positions,
        ),
    ]
)

blob.make_interchange()

/Users/mattthompson/mamba/envs/openff-toolkit-test/lib/python3.11/site-packages/openff/interchange/operations/_combine.py:104: InterchangeCombinationWarning: Interchange object combination is complex and may produce strange results outside of use cases it has been tested in. Use with caution and thoroughly validate results!
  warnings.warn(
/Users/mattthompson/mamba/envs/openff-toolkit-test/lib/python3.11/site-packages/openff/interchange/operations/_combine.py:84: InterchangeCombinationWarning: Found electrostatics 1-4 scaling factors of 5/6 with slightly different rounding (0.833333 and 0.8333333333). This likely stems from OpenFF using more digits in rounding 1/1.2. The value of 0.8333333333 will be used, which may or may not introduce small errors. 
  warnings.warn(


Interchange with 7 collections, non-periodic topology with 2487 atoms.

In [43]:
from openmmforcefields.generators import GAFFTemplateGenerator

gaff_template_generator = GAFFTemplateGenerator(molecules=ligands)
gaff = openmm.app.ForceField()
gaff.registerTemplateGenerator(gaff_template_generator.generator)

blob = Blob(
    pairings = [
        ForceFieldTopologyPairing(
            forcefields=[gaff],
            topology=ligands[-1].to_topology(),
            positions=ligands[-1].conformers[0].to_openmm(),
        ),
        ForceFieldTopologyPairing(
            forcefields=[openff.toolkit.ForceField("openff_no_water_unconstrained-3.0.0-alpha0.offxml")],
            topology=openff.toolkit.Topology.from_pdb("protein.pdb"),
        ),
    ]
)


blob.pairings[0].topology.box_vectors = blob.pairings[1].topology.box_vectors

blob.make_interchange()

Interchange with 7 collections, periodic topology with 2487 atoms.